# CEG-WM HF transmission diagnostic

This output-free Notebook preserves the completed historical eight-cluster development HF transmission diagnostic. It is paused and is not authorized to run; do not use **Run all**. The currently authorized entrypoint is `hf_only_detector_directional_validation.ipynb`. The historical repository server owned dependency installation, basic GPU checks, model revision download, the real runtime and method calls, formal records, persistence, and internal result or diagnostic ZIP creation.

This historical diagnostic fitted no threshold and supports no FPR, candidate promotion, calibration, formal evaluation, baseline, or paper claim. Its immutable revision, run, and records must not be continued or mixed with the current HF detector directional validation.


In [ ]:
from google.colab import drive, userdata
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')


In [ ]:
REPOSITORY_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
EXECUTION_REVISION = 'af1eea8f55086b583e3e5e4a02586959983db70b'
RUN_ID = 'ceg_wm_hf_transmission_diagnostic_server_execution'
SESSION_ID = datetime.now(timezone.utc).strftime('colab_%Y%m%dt%H%M%S%fz')
CHECKOUT_ROOT = Path(f'/content/ceg_wm_hf_transmission_checkout_{SESSION_ID}')
DRIVE_MOUNT = Path('/content/drive').resolve()
DRIVE_ROOT = DRIVE_MOUNT / 'MyDrive' / 'CEG-WM' / 'hf_transmission_diagnostic'
PERSISTENT_ROOT = DRIVE_ROOT / 'persistent'
CACHE_ROOT = DRIVE_ROOT / 'cache'
EXPORT_BASE = DRIVE_ROOT / 'exports' / EXECUTION_REVISION / RUN_ID
EXPORT_ROOT = EXPORT_BASE / SESSION_ID
for required_root in (PERSISTENT_ROOT, CACHE_ROOT, EXPORT_BASE):
    required_root.mkdir(parents=True, exist_ok=True)
    assert DRIVE_MOUNT in required_root.resolve().parents
probe_path = PERSISTENT_ROOT / f'.write_probe_{SESSION_ID}'
with probe_path.open('x', encoding='utf-8') as probe:
    probe.write('hf transmission persistent root available\n')
probe_path.unlink()
secret_environment = os.environ.copy()
secret_environment['HF_TOKEN'] = userdata.get('HF_TOKEN')
secret_environment['CEG_WM_ROOT_KEY'] = userdata.get('CEG_WM_ROOT_KEY')
assert secret_environment['HF_TOKEN'] and secret_environment['CEG_WM_ROOT_KEY']


In [ ]:
CHECKOUT_ROOT.mkdir(parents=True, exist_ok=False)
subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'init'], check=True)
subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'remote', 'add', 'origin', REPOSITORY_URL], check=True)
subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'fetch', '--depth', '1', 'origin', EXECUTION_REVISION], check=True)
subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
observed_revision = subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
observed_status = subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'status', '--porcelain'], check=True, capture_output=True, text=True).stdout
assert observed_revision == EXECUTION_REVISION
assert observed_status == ''


In [ ]:
server_entrypoint = CHECKOUT_ROOT / 'scripts/experiment_execution/hf_transmission_diagnostic_server.py'
command = [
    sys.executable, str(server_entrypoint),
    '--repository-root', str(CHECKOUT_ROOT),
    '--expected-revision', EXECUTION_REVISION,
    '--persistent-root', str(PERSISTENT_ROOT),
    '--cache-root', str(CACHE_ROOT),
    '--run-id', RUN_ID,
    '--session-id', SESSION_ID,
]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=secret_environment)
assert process.stdout is not None
for log_line in process.stdout:
    print(log_line, end='')
server_exit_code = process.wait()
del secret_environment
receipt_source = PERSISTENT_ROOT / RUN_ID / 'server_receipts' / SESSION_ID / 'execution_receipt.json'
assert receipt_source.is_file(), 'server did not persist the execution receipt'
receipt = json.loads(receipt_source.read_text(encoding='utf-8'))
assert receipt['committed_revision'] == EXECUTION_REVISION
assert receipt['run_id'] == RUN_ID and receipt['session_id'] == SESSION_ID
assert receipt['exit_code'] == server_exit_code


In [ ]:
def file_sha256(path):
    digest = sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def copy_to_drive_export(source, destination, expected_sha256):
    source = Path(source)
    destination = Path(destination)
    if destination.exists():
        raise RuntimeError('Drive export destination already exists')
    shutil.copyfile(source, destination)
    if file_sha256(destination) != expected_sha256:
        raise RuntimeError('Drive export SHA-256 mismatch')
    return destination

artifact_source = Path(receipt['artifact_path']).resolve()
assert artifact_source.is_file() and not artifact_source.is_symlink()
assert PERSISTENT_ROOT.resolve() in artifact_source.parents
assert receipt['artifact_kind'] in {'hf_transmission_diagnostic_result', 'hf_transmission_diagnostic_failure'}
assert receipt['formal_tau_created'] is False
assert receipt['candidate_promoted'] is False
assert receipt['scientific_claims_supported'] is False
assert file_sha256(artifact_source) == receipt['artifact_sha256']
receipt_sha256 = file_sha256(receipt_source)
EXPORT_ROOT.mkdir(parents=True, exist_ok=False)
artifact_export = copy_to_drive_export(artifact_source, EXPORT_ROOT / artifact_source.name, receipt['artifact_sha256'])
receipt_export = copy_to_drive_export(receipt_source, EXPORT_ROOT / 'execution_receipt.json', receipt_sha256)
checksums_path = EXPORT_ROOT / 'SHA256SUMS'
with checksums_path.open('x', encoding='utf-8') as checksums:
    checksums.write(f"{receipt['artifact_sha256']}  {artifact_export.name}\n")
    checksums.write(f'{receipt_sha256}  {receipt_export.name}\n')
summary = {
    'artifact_kind': receipt['artifact_kind'],
    'artifact_path': str(artifact_export),
    'artifact_sha256': receipt['artifact_sha256'],
    'receipt_path': str(receipt_export),
    'receipt_sha256': receipt_sha256,
    'checksums_path': str(checksums_path),
    'committed_revision': EXECUTION_REVISION,
    'run_id': RUN_ID,
    'session_id': SESSION_ID,
    'committed_unit_count': receipt.get('committed_unit_count', 0),
    'termination_reason': receipt.get('termination_reason'),
    'directional_decision': receipt.get('directional_decision'),
}
print(json.dumps(summary, indent=2, sort_keys=True))
if server_exit_code != 0:
    raise RuntimeError('HF transmission diagnostic ended with a diagnostic; Drive export is preserved')
